# 03 · Agreement, adjudication → *your* gold set

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egumasa/lda2-final-template/blob/main/notebooks/03_annotate.ipynb)

The part no model can do for you, and the part the Q&A will ask about.

```
  01_build_pool_<track>  →  02_sample  →  02b_add_samples  →▶ 03_annotate  →  04_develop  →  05_test  →  06_report
```

| | |
|---|---|
| **Reads** | `data/gold/<track>_<group>_sample.json` and the sheet (both from 02) |
| **Writes** | `data/gold/<track>_<group>_gold.json`, and its `_dev.json` / `_test.json` split |

---

**Come here when both coders have finished.** Notebook 02 drew the sample and made the sheet; this one turns two people's labels into one gold set.

The published labels are somebody else's judgment. You re-annotated the sample blind; now you find out how far apart the two of you were, and argue out the rows you disagreed on.

What comes out is *your* gold set — and the disagreements tell you which label boundaries are genuinely fuzzy. That is what lets you say, later, whether a model's miss is the **model's** fault or the **scheme's**. Nothing else in the project can tell you that, and notebook 06 asks you for it directly.

It ends by drawing one more line: which of your annotated items you are allowed to *look at* while you write prompts, and which are held back for the number you report. One sheet, one adjudication, then a split — it costs no extra coding.

## Setup — run this first

This cell mounts your Google Drive and finds your group's shared folder, `lda2-final-template`. Everything the project produces — the pool, the gold set, your prompts, the outputs — is an ordinary file in there, which is what makes it survive the runtime resetting *and* lets the rest of your group see it.

**One member sets the folder up once:**

1. That member runs the `git clone` line this cell prints if the folder is missing, which puts it in their own Drive.
2. They share it with the group (right-click ▸ *Share*), with edit access.
3. Everyone else opens *Shared with me*, right-clicks the folder, and chooses **Add shortcut to Drive** ▸ *My Drive*.

Keep that shortcut's name exactly `lda2-final-template`. It is what makes the same path work for all of you — if Drive renames it to `lda2-final-template (1)`, this cell will not find it.

From then on, open notebooks from the folder itself (*File ▸ Open notebook ▸ Drive*) rather than from the GitHub badge, so you are working on your group's copy and not a fresh one.

**Looking inside a helper.** The functions this cell imports are defined in `scripts/`. Two ways to read one, both the same ones you used on Day 2:

- `help(save_json)` prints its first line — what to pass in and what comes back — and the description of each argument. Typing `save_json(` and pressing **Shift+Tab** shows the same thing in a pop-up.
- To read the code itself, open `scripts/pipeline.py` from the **Files** panel on the left. Colab lists the functions in that file down the side, so you can click straight to the one you want.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first. You are not expected to read it.
# ------------------------------------------------------------------
# This cell is plumbing, and it is the only cell in the project that is.
# It finds your group's shared folder in Google Drive, because everything
# this project keeps goes in there: a Colab runtime is wiped when it resets,
# and nobody else in your group can see inside it. Then it makes the
# project's own code importable. Run it and move on; nothing below asks you
# to have understood it.

FOLDER = "lda2-final-template"     # the shared folder, in every member's Drive

import os, sys

PROJECT = ".."                              # running locally: it is just above us

try:
    from google.colab import drive           # only exists inside Colab
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/" + FOLDER
    if not os.path.isdir(PROJECT):
        raise RuntimeError(
            "Could not find " + PROJECT + "\n\n"
            "Setting the folder up for your group? Run this in a new cell:\n"
            "  !git clone https://github.com/egumasa/lda2-final-template.git "
            + PROJECT + "\n"
            "then share the folder with the rest of your group.\n\n"
            "Someone else already did? Open Drive, find the folder under "
            "'Shared with me', right-click it, and choose 'Add shortcut to "
            "Drive'. Keep the name exactly " + FOLDER + ".")
    # Work inside the project folder, where the notebooks live.
    os.makedirs(PROJECT + "/notebooks", exist_ok=True)
    os.chdir(PROJECT + "/notebooks")

# scripts/ and config.py, by their real paths - so they are found from wherever
# this notebook happens to be working.
sys.path.append(PROJECT)
sys.path.append(PROJECT + "/scripts")

# Re-read config.yaml every time this cell runs. Without the reload, Python
# hands back the settings it read the FIRST time, and editing config.yaml
# would appear to do nothing until you restarted the runtime.
import importlib
import config
importlib.reload(config)

# Named one by one rather than with `import *`, so that every name a cell
# below uses can be traced back to the file it came from — config.yaml for
# these, scripts/ for the rest.
from config import (TRACK, GROUP, RUN, SEED, N_PER_CLASS, DEV, CODERS,
                    MEMBERS, LABELS_ORDER, TEMPERATURE, MODEL, ROOT, OUT_DIR,
                    POOL_PATH, DEMO_POOL_PATH, SAMPLE_PATH, GOLD_PATH,
                    SAMPLE_BEFORE_TOPUP_PATH, DEV_PATH, TEST_PATH,
                    DISAGREED_PATH, PRED_PATH, ROUNDS_PATH, NOTES_PATH,
                    TESTLOG_PATH, PROMPT_FILE, SHEET_PATH, TRIAGE_PATH,
                    CODER_TRIAGE_PATH, describe)

# The Google Sheets round trip is plumbing, so it is imported, and so is the
# dev/test split: stratifying and rounding are bookkeeping, not judgment.
from pipeline import (load_gold, label_set, save_json, split_dev_test,
                      plot_confusion_matrix)
from annotate import (remembered_sheet, load_coder_sheets, to_canonical,
                      compare_to_published, column, percent_agreement,
                      fleiss_kappa)

# The scoring itself comes from scikit-learn, by its own names. You checked your
# hand-built versions of these against it on Day 2 S6; these are those functions.
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# pandas, for the table the function you write in step 2 hands back.
import pandas as pd

# For the optional section between steps 2 and 3: the same triage notebook 06
# runs on the model's errors, with one word changed for two people.
from metrics import triage_counts
from pipeline import CODER_CATEGORIES

# `disagreements` is NOT imported. Step 2 asks you to write it, because the rule
# inside it — what counts as a disagreement — is a decision about your scheme
# rather than a fact about your data.

describe()                  # what this notebook is working on


> **Everything above comes from `config.yaml`** — one small file at the top of the repo, which you edit once as a group, and the only file in the plumbing you touch. That is deliberate: the seed that drew your sample has to be the seed you report, and five copies of a number in five notebooks is five chances for them to disagree. Your settings are also the filenames — `track: cars50`, `group: kimura`, `run: v1` means this notebook reads and writes `cars50_kimura_v1_...`. If the line it just printed is not your track, your group and your seed, fix `config.yaml` and re-run this cell.

## First — your sample, back from the file

Notebook 02 saved it, and this is the moment that was for. Days have passed, the runtime that drew the sample is long gone, and the person running this cell may not be the person who ran 02.

It matters that this is a **load and not a redraw**: the sheet your coders filled in was built from these exact forty items, and adjudication puts their labels back onto them one by one. `to_canonical` also uses this list to restore what the sheet does not carry — on `cars50` and `raamove`, the passage each sentence came from.

In [ ]:
sampled = load_gold(SAMPLE_PATH)
LABELS = label_set(sampled)

print(len(sampled), "items ·", LABELS)

# These still carry the PUBLISHED label. Ignore it for now — you compare
# against it in step 4, once your own labels are settled.


## Then — the annotation sheet, found again

Now we look up the sheet notebook 02 created, from the small file it wrote the link to. That file is why the link is not lost: whoever runs this notebook need not be the person who ran notebook 02, and need not still have that cell's output on screen.

If this prints *none saved yet*, notebook 02's sheet step has not been run — or was run by somebody whose copy of the folder is not this one.

In [ ]:
SHEET_ID = remembered_sheet(SHEET_PATH)

# Working on a sheet someone made before this file existed? Paste its URL (or
# just the long id from it) here instead:
# SHEET_ID = ""

print("sheet:", SHEET_ID or "-- none saved yet: run notebook 02 --")


## Step 1 — How far apart were your coders?

Each coder has their **own tab**, so the first thing to do is line them up side by side. `load_coder_sheets` reads one tab per name you give it and joins them by item id into a single table — one column per coder, plus `Final`.

> **Who annotated?** `CODERS` comes from `config.yaml`, so notebook 06 finds the same tabs. If a third coder joined, duplicate an **empty** tab in the sheet (right-click ▸ *Duplicate*), rename it `CoderC`, and add it there.

Then the measuring, and **that part is yours to assemble.** Here is everything available:

| Call | What it gives you |
|---|---|
| `column(rows, "CoderA")` | one coder's labels as a plain list, in row order |
| `percent_agreement(a, b)` | how often two coders matched, ignoring chance |
| `cohen_kappa_score(a, b)` | agreement corrected for chance, two coders |
| `cohen_kappa_score(a, b, weights="quadratic")` | the same, counting a near miss as a smaller error |
| `fleiss_kappa([a, b, c])` | one number for three or more coders |
| `confusion_matrix(a, b, labels=LABELS)` | which label **pairs** you disagree about |
| `plot_confusion_matrix(matrix, LABELS, title)` | that matrix, drawn |

The last three you met on Day 2 S6, under those names, when you checked your own precision, recall, F1 and κ against scikit-learn's.

**Which of them you owe is not a free choice, and it is settled before you run anything** — by how many coders you have, and by whether `PLAN.md` §3 says your labels are a scale:

| Your design | Report |
|---|---|
| two coders, labels with no order | percent agreement **and** Cohen's κ |
| two coders, labels on a scale | those two, **and** the weighted κ |
| three or more coders | percent agreement **and** Fleiss' κ, plus Cohen's κ per pair |

Read that off your own design and write the cell. Choosing the statistic after seeing which one flatters you is the one thing that would make the number meaningless — which is exactly why the rule above depends on nothing you are about to find out.

**Look at the matrix before step 2**, whichever numbers you report. The κ says how far apart you were; only the off-diagonal cells say *which pair of labels* you disagree about, and that pair is what you go back to the sheet to argue about. Looking at it changes what you do next, so there is nothing to protect yourself from.

**Write the numbers down now** — they are report section 1, and they do not survive a runtime reset. A κ around .8 is strong; around .4 means the scheme, not the annotators, is doing something wrong. Either is a reportable finding. A low κ you can explain beats a high one you cannot.

Run it once **every** coder's tab is filled in — rows that not everyone labelled are left out of the comparison. If two coders appear to have given every item the same label, somebody duplicated a tab that had already been filled in; that agreement is a copy rather than a measurement.

Stuck? `from answers import answer`, then `answer("agreement")` — after you have tried.

In [ ]:
# ══ STEP 1 · How far apart were your coders? ══════════════════════════════
# Reads one tab per coder and lines them up by item id, then measures how far
# apart they are — with whichever statistics your design calls for.
# Creates: rows, a_labels, b_labels

# ✏️ Percent agreement alone counts lucky agreement as if you had earned it.
#    The table above says which κ your design owes. Add it.

rows = load_coder_sheets(SHEET_ID, CODERS)   # one read per tab, merged by ID

a_labels = column(rows, CODERS[0])
b_labels = column(rows, CODERS[1])

percent_agreement(a_labels, b_labels)


**✍️ For your report and the Q&A** — this goes in the write-up, not in a cell below.

> Our coders agreed on ___% of items, with a ___ κ of ___.
>
> We report ___ as well as percent agreement because our labels ___.
>
> The pair we disagreed about most often was ___ and ___, which suggests our scheme ___.

The third sentence is the one the Q&A goes to. It comes off the matrix, and it is a claim about your **scheme** rather than about your coders — two labels that keep swapping usually means the boundary between them is not written down clearly enough.

A low κ you can explain beats a high one you cannot.

## Step 2 — What counts as a disagreement?

Now you need the list of rows to argue about. **This one you write**, and it is the only function in the project that you do — because the rule inside it is a decision about your scheme rather than a fact about your data, and there is no way to hand it over without answering it for you.

The obvious rule: a row is a disagreement when your coders did not all choose the same label. For most schemes that is the right one.

It is not the only defensible one. **If your labels sit on a scale** — A1 < A2 < … < C2, Low < Mid < High — you might decide that neighbouring labels are two people reading the same sentence much the same way, and that only a gap of two or more is worth an argument. That version hands back a shorter list and sends you to the sheet with less to settle. Which you chose, and why, is a sentence in your report; not knowing which you used is the only wrong answer.

You have written this shape before. It is a loop over `rows`, keeping the ones where the labels differ:

```python
def disagreements(rows, coders):
    out = []
    for row in rows:
        ...        # pull each coder's label out of this row
        ...        # keep the row if they are not all the same
    return pd.DataFrame(out, columns=list(rows[0]))
```

The cell below it calls yours as `disagreements(rows, coders=CODERS)` — by keyword, so that it works whether your second parameter is named `coders` or you pasted the reference version, whose first two parameters are the two column names.

`column(rows, name)` is not what you want inside here — that reads a whole column down the sheet, and you are working across one row at a time. `row.get(name, "")` is the piece you need, and `str(...).strip()` around it drops the stray spaces a spreadsheet loves to add.

**Leave out the rows nobody finished.** A blank cell is a coder who has not got there yet, not two people disagreeing, and counting it as a disagreement puts an item on your adjudication list that has nothing to adjudicate.

Tried it? `from answers import answer`, then `answer("disagreements")`.

In [ ]:
# ══ STEP 2 · Write the rule ═══════════════════════════════════════════════
# Defines the function that decides which rows go on your adjudication list.
# Nothing runs until the cell below calls it.
# Creates: disagreements

# ✏️ your code here


### Now run it, and keep the result

This cell calls the function you just wrote and saves what comes back.

The table comes back in notebook 06, where the rows your coders argued about are what you check the model's errors against — and saving it here means 05 does not have to sign back in to the sheet and derive the same table a second time. It also means that step still works after the sheet has been deleted, or its owner has left.

The last line is just the name `disagreed`, with no `print`. In a notebook the value of a cell's last line is displayed automatically, and for a table that reads far better than `print` would. Add a line after it and the table stops appearing, which is the one thing to watch out for.

In [ ]:
# `coders=` by name, not by position: the reference version in answers.py
# takes the two column names first, so a pasted copy of it would read CODERS
# as one coder's name if this were positional.
disagreed = disagreements(rows, coders=CODERS)
save_json(disagreed.to_dict("records"), DISAGREED_PATH,
          what="rows your coders disagreed on")
disagreed

**✍️ For your report and the Q&A** — this goes in the write-up, not in a cell below.

> We counted a row as a disagreement when ___.
>
> That put ___ of ___ items on our adjudication list.
>
> We chose that rule rather than ___ because ___.

Both rules are defensible and they hand you different lists, so the sentence that matters is the third one. If your labels are not on a scale there was only one sensible rule, and saying so is a complete answer.

---

## Optional, before you adjudicate — what *kind* of disagreement is each one?

**This section is worth about ten minutes and you may skip it.** If your list is short, or the session is running late, go straight to step 3. Nothing below it depends on this.

Here is what it buys. You are about to sit down and settle every row on that list. While you argue each one out you will form an opinion about *why* the two of you differed — and that opinion is gone by the afternoon unless you write it down as you go. It answers two things nothing else in the project computes:

- **"What did your QC pass change?"** is a question published in advance for the Q&A, and the rubric asks what your adjudication changed. *"Six of our fourteen disagreements were the scheme's fault"* is an answer; *"we discussed them"* is not.
- **Whether a second annotation round is worth the afternoon** — the decision the next section puts in front of you.

Four words, and they are the ones notebook 06 uses on the **model's** errors, with one changed. The question is the same one — whose fault is this? — asked about two people instead of about a model:

| Word | What it means |
|---|---|
| **`scheme`** | the item is genuinely borderline under your scheme. The boundary is not written down clearly enough |
| **`wording`** | the *label name* misled one of you. `Gap` reads as "missing data" to somebody who has not read the guidelines twice |
| **`ambiguous`** | the item itself is unclear in a way no scheme would settle |
| **`slip`** | one of you misread the sentence or clicked the wrong cell. Not evidence about anything |

`slip` is where `model` sits in notebook 06's list, and the swap is the point: between two people there is no model to blame, and a mis-click is not a finding.

Do it **out loud, together**, with the sentences in front of you — one line per row as you settle it.

In [ ]:
# One line per disagreed row: the id, then the category word, then WHY.
# It runs empty and tells you how many rows you still owe, so add lines and
# re-run as you work down the list.
CODER_TRIAGE = {
    # 7: "scheme — where Move 1 ends and Move 2 starts is not written down",
    # 12: "slip — we both meant A2; one of us typed in the wrong row",
}

# `categories=` is what makes this the CODER list rather than the model one.
triage_counts(CODER_TRIAGE, disagreed, categories=CODER_CATEGORIES,
              what="disagreements")

### Save it, then read the counts as a decision

Notebook 06 opens this file and asks a sharper question than *"did the model err where we argued?"* — it asks whether the model missed **the items you yourselves called `scheme`**. Those are the ones you have independent evidence about.

In [ ]:
save_json(CODER_TRIAGE, CODER_TRIAGE_PATH,
          what="your reading of the coder disagreements")

## Optional — a second annotation round

Now read the counts as a decision, because they point two different ways:

| What you found | What it is worth doing |
|---|---|
| mostly **`scheme`** or **`wording`** | the disagreements are your guidelines' fault, and rewriting the boundary rule then re-annotating will genuinely move κ. This is what a second round is for. |
| mostly **`slip`** or **`ambiguous`** | a second round would re-measure the same fuzziness. Adjudicate, note the ambiguity as a limitation, and move on. |

**A second round is not expected and not required.** It costs another pass over every item, and a group that spends the time on error analysis instead has made a defensible choice. What is *not* defensible is a low κ with no account of why.

If you do one, the mechanics are three moves in the sheet and one line here:

1. **Rewrite the guideline first.** A second round with the same scheme is the same round. Add the rule, the boundary case, or the example that would have settled the rows you just called `scheme`.
2. **In the sheet:** right-click `CoderA` ▸ *Duplicate*, rename the copy `round2_CoderA`, and **clear the `Label` column** in it. Same for `CoderB`. The originals stay untouched, which is what lets you report both rounds.
3. **Re-annotate** those two tabs, blind, against the rewritten guideline.

Then run the cell below. It prints the two tab names; put them in `config.yaml` under `coders:` and re-run this notebook **from the top**. Everything works unchanged, because `load_coder_sheets` takes tab names and `config.yaml` is where they come from.

Report both κ values and say what you changed between them. A κ that moved from .41 to .68 because you rewrote one boundary rule is one of the strongest things your report can say — and your original tabs are untouched, which is what lets you report both rounds rather than only the better one.

In [ ]:
# The tab names for round 2. `load_coder_sheets` takes TAB NAMES, so nothing
# else in this notebook has to change — but the names belong in config.yaml
# rather than in a variable here.
#
# Assigning to CODERS in this cell would look like it worked and then quietly
# undo itself: re-running the notebook from the top re-runs SETUP, which reads
# CODERS back out of config.yaml. Step 1 would measure round 1 again while you
# believed you were reading round 2, and nothing would say so.
round2 = []
for name in CODERS:
    round2.append("round2_" + name)   # CoderA -> round2_CoderA

print("Put these in config.yaml under `coders:`, then re-run this notebook")
print("from the top:")
for name in round2:
    print("  -", name)

**✍️ For your report and the Q&A** — this goes in the write-up, not in a cell below.

> Of ___ disagreements, ___ were `scheme`, ___ `wording`, ___ `ambiguous` and ___ `slip`.
>
> We did / did not run a second annotation round, because ___.
>
> Our κ went from ___ to ___ after we ___.

The first sentence is what "what did your QC pass change?" is asking for. The second is a real decision either way — the reason is what is graded, not which way you went. Drop the third if you ran one round.

Working out the counts by hand is fine if you skipped the cell above.

## Step 3 — Adjudicate

Go back to the sheet and fill in `Final` for **every** row:

- Where you agreed, `Final` is that label.
- Where you did not, talk it out and decide. If you cannot agree, the scheme is underspecified — write down *why* in `Note` and pick one. That note is worth more to your report than the label is.

Then re-read the sheet and canonicalise it. `to_canonical` reports blanks and invalid labels rather than silently dropping them; fix them in the sheet and re-run until it says **0 blank, 0 invalid**. A blank row is an item that has gone missing from your study without telling you.

The cell re-reads the sheet first, because `rows` from step 1 was fetched before you filled `Final` in. And it passes `source=sampled`: gold is rebuilt from the **sheet**, which carries only the id, the text and your label, so anything else the item had — on `cars50` and `raamove`, its passage — is put back from `sampled` by id. On the other tracks that argument does nothing.

Both calls are the Day 2 S5 step F ones. `to_canonical` is in `scripts/annotate.py`.

In [ ]:
# ══ STEP 3 · Adjudicate, then canonicalise ════════════════════════════════
# Re-reads the sheet now that Final is filled in, and turns it into your gold
# set — reporting any row that is blank or has a label it does not recognise.
# Creates: gold

# Re-read: `rows` from step 1 was fetched before you filled in Final.
rows = load_coder_sheets(SHEET_ID, CODERS)

gold = to_canonical(rows, LABELS, source=sampled)   # re-attaches what the sheet drops


## Step 4 — Where do you differ from the published labels?

Now — and only now, with your own labels settled — look at what the corpus said. `compare_to_published` matches by text and shows you every row where your group landed somewhere else.

**Disagreement here is not an error.** You annotated forty items carefully against a scheme you had thought about; the original annotators worked at scale under different guidelines. Where you differ, one of three things is true, and saying which is exactly the analytical work this project is for:

1. **Your scheme drifted** from theirs — you read a category boundary differently. Say where.
2. **The item is genuinely ambiguous** — it would split any pair of annotators.
3. **One of you is wrong.** It happens, in both directions.

This table is report section 1, and it is the one that most often produces a sentence worth saying out loud in the Q&A. Pick two or three rows and write down which of the three cases above they are — now, while you still remember the argument you had about them.

The comparison runs against `sampled`, not `pool`: sampling renumbered the ids, so `pool` would line your item 7 up against a completely different sentence. This is the Day 2 S5 step F call; it is in `scripts/annotate.py`.

In [ ]:
# ══ STEP 4 · Compare against the published labels ═════════════════════════
# Shows every row where your group's label and the corpus's label differ.
# Creates: differences

differences = compare_to_published(gold, sampled)   # sampled, not pool: same 40 items
differences


## Save it — this is the handoff

This file is the single most valuable thing your group makes all week — hours of judgment, and the only thing in the project that could not have been produced by a script. Every number in notebooks 04, 05 and 06 is measured against it, and it goes in your submission bundle.

**Next:** open `04_develop.ipynb`. It starts by loading `data/gold/<track>_<group>_gold.json`.

In [ ]:
save_json(gold, GOLD_PATH, what="gold items")

# It is git-ignored — it is your work, not part of the template. If you cloned
# into Google Drive it is already saved across sessions; if not, download it.


## Step 5 — Draw the line: dev and test

In notebook 04 you will change your prompt because of what you saw it get wrong. That is the work. But a score measured on the same items you kept adjusting against stops being a measure of how good your prompt is, and becomes a measure of **how long you kept adjusting**. It only ever goes up.

So the line gets drawn now, before anything has been run against these items:

| | what it is for |
|---|---|
| **dev** | the items you may look at. Iterate here, as many rounds as you like. |
| **test** | opened once, in `05_test.ipynb`. Whatever it says is what you report. |

Both halves came out of the same sheet and the same adjudication, so the split costs you no extra annotation. What it costs is items you are allowed to learn from — which is why the ratio is a real decision and `PLAN.md` §6 asks you to defend the one you made. A bigger dev gives steadier feedback while you iterate and leaves a smaller test, so the number you finally report bounces more; a smaller dev means prompt decisions made on very few items, which is how you tune to noise and then watch the gain evaporate.

This also replaces the old advice to keep `n_per_class` at 2 while iterating. **dev is the fast set now** — a dozen or so items is about a minute per round, and your sample stays at full size throughout.

The split is stratified by label, so both halves keep every label wherever the data allows. Where it does not — a label with a single surviving item — that item goes to **test**, and the function says so. That asymmetry is deliberate: a label missing from test drops out of the macro average without announcing itself, while a label missing from dev only costs you feedback.

### What `split_dev_test` does with your `dev:` setting

It is imported rather than printed here, because what is inside it is bookkeeping — grouping by label, rounding a fraction to a whole number of items, and being careful about a label with only one item left. None of that is a decision you make; the ratio is, and that is in `config.yaml`.

Three things it does that are worth knowing, because they show up in your numbers:

- **A rare class goes to test, not dev.** A label with a single surviving item cannot be on both sides. Missing from test, it drops out of your macro average without announcing itself; missing from dev, it only costs you feedback. The second is the cheaper mistake, so that is the one it makes.
- **The rounding is written out** rather than left to `round()`, which in Python rounds 0.5 down and 1.5 up — not something you want to explain in the Q&A.
- **The ids are not renumbered.** Notebook 06 asks which of the model's errors are also the rows your coders argued about, and that join runs on these ids.

`help(split_dev_test)` prints what to pass it. To read the code itself, run `split_dev_test??` — it prints the source of any function, imported or not.

Now we draw the line: which of your gold items you are allowed to look at while iterating, and which you are not. Nothing here existed in Days 1–3 — no set there was worth holding back. `split_dev_test` is the function you defined and read just above.

**Run this once, and before you open notebook 04.** Splitting again after you have iterated on dev means the held-out items have already been seen — by you, if not by the model.

How big dev is comes from `dev:` in `config.yaml`, and how you wrote the number says what you meant. A balanced draw (`sample_pool`) suits a whole number, `dev: 3` — three items per label. An uneven one (`sample_random`) suits a decimal, `dev: 0.35` — a third of each label, because a fixed 3 per class would eat a small class whole.

Nothing is saved yet — read the counts it prints first.

In [ ]:
# ══ STEP 5 · Split dev / test ═════════════════════════════════════════════
# Splits your gold set in two, keeping every label on both sides wherever the
# data allows, and prints how many items each half got.
# Creates: dev, test

# DEV comes from config.yaml: a whole number is items per label, a decimal
# is a proportion of each label.
#
# Drew your sample with sample_by_document? Add by_document=True inside the
# brackets below, so that no passage has some of its sentences in dev and
# the rest in test.
dev, test = split_dev_test(gold, DEV, seed=SEED)


### Now save both halves

Read the per-label counts the split just printed **before** you run this. A label that lands in dev but not in test cannot appear in the score you report, and this is the last easy moment to change `dev` in `config.yaml` and draw the line again.

Once you are happy, save. Notebook 04 opens `dev`; notebook 05 opens `test`, once.

In [ ]:
save_json(dev,  DEV_PATH,  what="dev items")
save_json(test, TEST_PATH, what="test items")

**✍️ For your report and the Q&A** — this goes in the write-up, not in a cell below.

> We split ___ gold items into ___ dev and ___ test.
>
> We set `dev:` to ___ because ___.
>
> Every label is present on both sides except ___.

There is no right ratio at this size, only one you can defend, so the second sentence is the whole answer. The third is the limitation a reader needs in order to read your per-label scores — a label that is only in test was never something you could iterate against.

---

## 🛑 The `PLAN.md` gate

Notebook 04 starts calling the model. **Do not open it until your `PLAN.md` has been read and signed off.** It takes two minutes and it is not busywork: a mismatched label set or an unstated sampling seed costs an hour to unpick *after* you have burned quota on it.

Check, out loud, that these three agree: the label set in `PLAN.md`, the labels `label_set` actually returned above, and the labels your prompt file names. And that `PLAN.md` records **which sampling strategy you chose, and why**, and **§6: which split spec you set, the sizes it produced, and why that ratio**.